In [206]:
#importing libraries and dataframe
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from  sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('listings_cleaned.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head(10)

Shape: (53355, 61)


,listing_price,bedrooms,bathrooms,square_feet_num,number_of_storeys,latitude,longitude,lot_size_sqft,taxes,median_income,...,city_Peel,city_Pickering,city_Richmond Hill,city_Scugog,city_Toronto,city_Uxbridge,city_Vaughan,city_Whitby,city_Whitchurch-Stouffville,city_York
0,650000,2.0,2.0,649.5,14.0,43.708920,-79.392624,3633.0000,3167.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
1,349000,1.0,1.0,749.5,4.0,43.707298,-79.427315,3633.0000,1518.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
2,1200000,4.0,2.0,1300.0,2.0,43.680000,-79.428429,2275.0878,7134.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
3,769000,4.0,3.0,1899.5,2.0,43.712524,-79.276566,3633.0000,3951.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
4,379900,4.0,3.0,1899.5,2.0,43.712524,-79.276566,3633.0000,3951.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
5,309900,4.0,3.0,1899.5,2.0,43.712524,-79.276566,3633.0000,3951.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
6,1498000,6.0,5.0,900.0,1.0,43.792427,-79.397087,6737.5000,6956.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
7,1428000,6.0,5.0,900.0,1.0,43.792427,-79.397087,6737.5000,6956.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
8,349900,6.0,5.0,900.0,1.0,43.792427,-79.397087,6737.5000,6956.0,58381.0,...,False,False,False,False,True,False,False,False,False,False
9,649900,5.0,4.0,2750.0,2.0,43.708344,-79.387024,4917.4846,15610.0,58381.0,...,False,False,False,False,True,False,False,False,False,False


In [171]:
df.isnull().sum()

listing_price                  0
bedrooms                       0
bathrooms                      0
square_feet_num                0
number_of_storeys              0
                              ..
city_Uxbridge                  0
city_Vaughan                   0
city_Whitby                    0
city_Whitchurch-Stouffville    0
city_York                      0
Length: 61, dtype: int64

In [172]:
nan_cols = df.isnull().sum()
print(nan_cols[nan_cols > 0])

latitude             55
longitude            56
median_income     20032
population        20032
total_rooms         199
bath_bed_ratio      199
listing_year         80
listing_month        80
dtype: int64


In [173]:
print(df['days_since_sold'].describe())
print(df[['listing_year', 'listing_month', 'days_since_sold']].head(10))

count    53355.000000
mean       262.915303
std         42.397808
min          0.000000
25%        265.000000
50%        265.000000
75%        265.000000
max        365.000000
Name: days_since_sold, dtype: float64
   listing_year  listing_month  days_since_sold
0        2026.0            3.0            265.0
1        2026.0            1.0            265.0
2        2026.0            4.0            265.0
3        2026.0            3.0            265.0
4        2011.0            6.0            265.0
5        2005.0            2.0            265.0
6        2025.0           10.0            265.0
7        2020.0           11.0            265.0
8        2004.0            5.0            265.0
9        2004.0           10.0            265.0


In [174]:
mask = df['total_rooms'].isna()
df.loc[mask, 'total_rooms'] = df.loc[mask, 'bedrooms'] + df.loc[mask, 'bathrooms']

In [175]:
df.shape

(53355, 61)

In [176]:
city_cols = [c for c in df.columns if c.startswith('city_')]
df['_city'] = df[city_cols].idxmax(axis = 1).str.replace('city_' ,"",regex = False)

print('% missing median_income per city:')
missing_by_city = df.groupby('_city')['median_income'].apply(
    lambda x: x.isna().mean() * 100
).round(1).sort_values(ascending=False)
print(missing_by_city)

print('% missing population  per city:')
missing_by_city_pop = df.groupby('_city')['population'].apply(
    lambda x: x.isna().mean() * 100
).round(1).sort_values(ascending=False)
print(missing_by_city_pop)

% missing median_income per city:
_city
Toronto                   68.6
Aurora                     0.0
Brock                      0.0
Burlington                 0.0
Caledon                    0.0
Clarington                 0.0
Durham                     0.0
Georgina                   0.0
Halton                     0.0
Brampton                   0.0
Halton Hills               0.0
King                       0.0
Milton                     0.0
Markham                    0.0
Oakville                   0.0
Oshawa                     0.0
Peel                       0.0
Mississauga                0.0
Pickering                  0.0
Richmond Hill              0.0
Scugog                     0.0
Uxbridge                   0.0
Vaughan                    0.0
Whitby                     0.0
Whitchurch-Stouffville     0.0
York                       0.0
Name: median_income, dtype: float64
% missing population  per city:
_city
Toronto                   68.6
Aurora                     0.0
Brock             

In [177]:
toronto_mask = df['_city'] == 'Toronto'
print("toronto rows:", toronto_mask.sum())
print("toronto rows with median_income known:", df[toronto_mask]['median_income'].notna().sum())
print("toronto rows with population known:", df[toronto_mask]['population'].notna().sum())

print('\n Toronto median_income stats (non-null):')
print(df[toronto_mask]['median_income'].describe())

toronto rows: 29216
toronto rows with median_income known: 9184
toronto rows with population known: 9184

 Toronto median_income stats (non-null):
count     9184.0
mean     58381.0
std          0.0
min      58381.0
25%      58381.0
50%      58381.0
75%      58381.0
max      58381.0
Name: median_income, dtype: float64


In [178]:
toronto_incomes = df[df['_city']== 'Toronto']['median_income'].dropna().iloc[0]
toronto_population = df[df['_city']== 'Toronto']['population'].dropna().iloc[0]

print(f"Toronto median_income constant: ${toronto_incomes:,.0f}" )
print(f"Toronto population constant: ${toronto_population:,.0f}" )

toronto_mask = df['_city'] == 'Toronto'
df.loc[toronto_mask, 'median_income'] = df.loc[toronto_mask, 'median_income'].fillna(toronto_incomes)
df.loc[toronto_mask, 'population'] = df.loc[toronto_mask, 'population'].fillna(toronto_population)

print(f"\n Remaining NaNs in median_income: {df['median_income'].isna().sum()}")
print(f"\n Remaining NaNs in population : {df['population'].isna().sum()}")

Toronto median_income constant: $58,381
Toronto population constant: $2,600,000

 Remaining NaNs in median_income: 0

 Remaining NaNs in population : 0


In [197]:
df = df.dropna(subset = ['latitude', 'longitude'])

reference_date = pd.Timestamp('2025-01-01')
df["_sale_date"] = reference_date - pd.to_timedelta(df['days_since_sold'], unit = "D")
mask = df['listing_year'].isna()
df.loc[mask, 'listing_year'] = df.loc[mask, '_sale_date'].dt.year
df.loc[mask, 'listing_month'] = df.loc[mask, '_sale_date'].dt.month
df = df.drop(columns = ['_sale_date'])

mask = df['bath_bed_ratio'].isna()
df.loc[mask, 'bath_bed_ratio'] = df.loc[mask, 'bathrooms'] / df.loc[mask, 'bedrooms']

print(f"Remaining NaN in df: {df.isnull().sum().sum()}")
print(f"Shape after fixes: {df.shape}")

df_eng = df.copy()
df_v2 = df.copy()
print(f"Remaining NaN in df_eng: {df_eng.isnull().sum().sum()}")
print(f"Shape after fixes: {df_eng.shape}")

Remaining NaN in df: 0
Shape after fixes: (53299, 60)
Remaining NaN in df_eng: 0
Shape after fixes: (53299, 60)


In [180]:
print(df['days_since_sold'].describe())

count    53299.000000
mean       262.913113
std         42.420022
min          0.000000
25%        265.000000
50%        265.000000
75%        265.000000
max        365.000000
Name: days_since_sold, dtype: float64


In [181]:
# Listing Key columns 

KEY_COLS = ['bedrooms', 'bathrooms', 'square_feet_num', 'number_of_storeys', 'latitude', 
            'longitude', 'lot_size_sqft', 'taxes', 'median_income', 'population', 'parking_spaces', 
            'total_rooms', 'bath_bed_ratio', 'listing_year', 'listing_month', 'bedrooms_missing', 'bathrooms_missing',
            'square_feet_num_missing', 'number_of_storeys_missing', 'lot_size_sqft_missing', 'taxes_missing',
            'parking_spaces_missing', 'days_since_sold', 'house_age_encoded', 'property_type_Common Element Condo', 
            'property_type_Condo Apartment', 'property_type_Condo Townhouse', 'property_type_Detached', 'property_type_Link',
            'property_type_Other', 'property_type_Semi-Detached', 'property_type_Vacant Land', 'city_Aurora', 'city_Brampton', 
            'city_Brock', 'city_Burlington', 'city_Caledon', 'city_Clarington', 'city_Durham', 'city_Georgina', 'city_Halton',
            'city_Halton Hills', 'city_King', 'city_Markham', 'city_Milton', 'city_Mississauga', 'city_Oakville', 'city_Oshawa', 
            'city_Peel', 'city_Pickering', 'city_Richmond Hill', 
            'city_Scugog', 'city_Toronto', 'city_Uxbridge', 'city_Vaughan', 'city_Whitby', 'city_Whitchurch-Stouffville', 'city_York']

In [182]:
df = df.drop(columns = ['_city', 'house_age'])


X = df[KEY_COLS]
Y = df['sold_price']

x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size = 0.2, random_state=42)

print(f"x_train.shape: {x_train.shape}")
print(f"x_test.shape: {x_test.shape}")
print(f"NaNs in x_train: {x_train.isnull().sum().sum()}")
print(f"NaNs in x_train: {x_test.isnull().sum().sum()}")

x_train.shape: (42639, 58)
x_test.shape: (10660, 58)
NaNs in x_train: 0
NaNs in x_train: 0


In [183]:
#baseline OLS model
ols = LinearRegression()
ols.fit(x_train, y_train)

y_pred = ols.predict(x_test)

mape = mean_absolute_percentage_error(y_test, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("OlS model results")
print(f"MAPE {mape:.2f}%")
print(f"RMSE {rmse:,.0f}%")
print(f"r^2: {r2:.4f}")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if mape <= 15 else 'Fail'}")

OlS model results
MAPE 23.32%
RMSE 265,600%
r^2: 0.6509
Target 10% deviation (Zolo benchmark): Fail
Target 15% deviation (sold price): Fail


In [139]:
print(x_train.dtypes[x_train.dtypes == 'object'])

inf_cols = np.isinf(x_train.select_dtypes(include = np.number)).sum()
print(inf_cols[inf_cols >  0])

nan_cols = x_train.isnull().sum()
print(nan_cols[nan_cols > 0])

Series([], dtype: object)
Series([], dtype: int64)
Series([], dtype: int64)


In [141]:
x_train_vif = x_train.astype(np.float64)

vif_data = pd.DataFrame()
vif_data['feature'] = x_train_vif.columns
vif_data["VIF"] = [variance_inflation_factor(x_train_vif.values, i)
                        for i in range(x_train_vif.shape[1])]

vif_data = vif_data.sort_values("VIF", ascending = False).reset_index(drop= True)

print("VIF results")
print(vif_data.to_string())


VIF results
                               feature           VIF
0                             bedrooms           inf
1                            bathrooms           inf
2                          total_rooms           inf
3                         listing_year  1.139519e+05
4                            longitude  6.931451e+04
5                             latitude  6.762132e+04
6                         city_Toronto  6.769339e+03
7                            city_Peel  1.467092e+03
8                            city_York  1.395557e+03
9                          city_Durham  1.145164e+03
10                       taxes_missing  1.023916e+03
11                          population  8.804985e+02
12                         city_Halton  8.212756e+02
13             square_feet_num_missing  6.712446e+02
14              parking_spaces_missing  3.165826e+02
15                       median_income  1.994526e+02
16                      bath_bed_ratio  9.892652e+01
17           number_of_storeys_mis

In [142]:
drop_vif =  ['total_rooms', 'bathrooms', 'listing_year', 
             'latitude', 'longitude', 
             'population', 'median_income', 'days_since_sold']

x_train_reduced = x_train.drop(columns = drop_vif)
x_test_reduced = x_test.drop(columns = drop_vif)

print(f"features before: {x_train.shape[1]}")
print(f"features after: {x_train_reduced.shape[1]}")

features before: 58
features after: 50


In [144]:
#reduced baseline OLS model
ols_reduced = LinearRegression()
ols_reduced.fit(x_train_reduced, y_train)

y_pred_reduced = ols_reduced.predict(x_test_reduced)

mape = mean_absolute_percentage_error(y_test, y_pred_reduced) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred_reduced))
r2 = r2_score(y_test, y_pred_reduced)

print("reduced OLS model results")
print(f"MAPE {mape:.2f}%")
print(f"RMSE {rmse:,.0f}%")
print(f"r^2: {r2:.4f}")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if mape <= 15 else 'Fail'}")

reduced OLS model results
MAPE 37.24%
RMSE 343,787%
r^2: 0.4151
Target 10% deviation (Zolo benchmark): Fail
Target 15% deviation (sold price): Fail


In [ ]:
#vid version two 
drop_vif =  ['total_rooms', 'listing_year', 'days_since_sold']

x_train_reduced_version_two = x_train.drop(columns = drop_vif)
x_test_reduced_version_two = x_test.drop(columns = drop_vif)

print(f"features before: {x_train.shape[1]}")
print(f"features after: {x_train_reduced_version_two.shape[1]}")

features before: 58
features after: 55


In [ ]:
#reduced baseline OLS model version 2
ols_reduced = LinearRegression()
ols_reduced.fit(x_train_reduced_version_two, y_train)

y_pred_reduced_version_two = ols_reduced.predict(x_test_reduced_version_two)

mape = mean_absolute_percentage_error(y_test, y_pred_reduced_version_two) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred_reduced_version_two))
r2 = r2_score(y_test, y_pred_reduced_version_two)

print("reduced OLS model results")
print(f"MAPE {mape:.2f}%")
print(f"RMSE {rmse:,.0f}%")
print(f"r^2: {r2:.4f}")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if mape <= 15 else 'Fail'}")

reduced OLS model results
MAPE 36.48%
RMSE 339,560%
r^2: 0.4294
Target 10% deviation (Zolo benchmark): Fail
Target 15% deviation (sold price): Fail


In [194]:
#test 3: Ridge with full featture set
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

alphas_ridge = [0.1, 1.0, 10.0, 100.0, 1000.0]

print(f"{'Alpha':<10} {'MAPE':<12} {'RMSE':<15} {'R^2':<10}")
print("-" * 47)

for alpha in alphas_ridge:
    ridge = Ridge(alpha=alpha)
    ridge.fit(x_train_scaled, y_train)
    y_pred_ridge = ridge.predict(x_test_scaled)

    mape = mean_absolute_percentage_error(y_test, y_pred_ridge) * 100
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
    r2 = r2_score(y_test, y_pred_ridge)

    print(f" {alpha:<10} {mape:<12.2f} ${rmse:<14,.0f} {r2: <10.4f}")

Alpha      MAPE         RMSE            R^2       
-----------------------------------------------
 0.1        23.32        $265,600        0.6509    
 1.0        23.32        $265,603        0.6509    
 10.0       23.32        $265,628        0.6508    
 100.0      23.30        $265,776        0.6505    
 1000.0     23.12        $265,990        0.6499    


Tried ridge regression to deal with collinearity, trying lasso regression

In [153]:
alphas_lasso = [0.1, 1.0, 10.0, 100.0, 1000.0]

print(f"{'Alpha':<10} {'MAPE':<12} {'RMSE':<15} {'R^2':<10} {'features Kept': <15}")
print("-" * 62)

for alpha in alphas_lasso:
    lasso = Lasso(alpha = alpha, max_iter = 10000)
    lasso.fit(x_train_scaled, y_train)
    y_pred_lasso = lasso.predict(x_test_scaled)

    mape = mean_absolute_percentage_error(y_test, y_pred_lasso) * 100
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    r2 = r2_score(y_test, y_pred_lasso)
    features_kept = np.sum(lasso.coef_ !=0)

    print(f" {alpha:<10} {mape:<12.2f} ${rmse:<14,.0f} {r2: <10.4f} {features_kept: <15}")

Alpha      MAPE         RMSE            R^2        features Kept  
--------------------------------------------------------------
 0.1        23.32        $265,600        0.6509     58             
 1.0        23.32        $265,601        0.6509     58             
 10.0       23.32        $265,607        0.6509     57             
 100.0      23.31        $265,689        0.6507     56             
 1000.0     23.25        $265,964        0.6500     44             


tested for collinearity and it wasn't the problem with the model

In [156]:
def backward_elimination(x, y, significance_level = 0.05):
    features = list(x.columns)
    while True:
        x_with_const = sm.add_constant(x[features].astype(np.float64))
        model = sm.OLS(y, x_with_const).fit()

        pvalues = model.pvalues.drop('const')
        max_pvalue = pvalues.max()

        if max_pvalue > significance_level:
            removed_feature = pvalues.idxmax()
            features.remove(removed_feature)
            print(f"Removed: {removed_feature:<40} p-value: {max_pvalue:.4f}")
        else:
            break
    print(f"\n final features kept: {len(features)}")
    return features

lasso_best = Lasso(alpha =1000, max_iter = 10000)
lasso_best.fit(x_train_scaled, y_train)

lasso_kept_features = list(x_train.columns[lasso_best.coef_ !=0])
print(f"Lasso kept {len(lasso_kept_features)} features:")
print(lasso_kept_features)

x_train_be = x_train[lasso_kept_features].astype(np.float64)
selected_features = backward_elimination(x_train[lasso_kept_features], y_train)


Lasso kept 44 features:
['bathrooms', 'square_feet_num', 'number_of_storeys', 'latitude', 'longitude', 'lot_size_sqft', 'taxes', 'median_income', 'population', 'parking_spaces', 'total_rooms', 'bath_bed_ratio', 'listing_year', 'listing_month', 'bedrooms_missing', 'bathrooms_missing', 'lot_size_sqft_missing', 'taxes_missing', 'days_since_sold', 'house_age_encoded', 'property_type_Common Element Condo', 'property_type_Condo Apartment', 'property_type_Condo Townhouse', 'property_type_Detached', 'property_type_Link', 'property_type_Other', 'property_type_Semi-Detached', 'property_type_Vacant Land', 'city_Brampton', 'city_Burlington', 'city_Caledon', 'city_Durham', 'city_Halton Hills', 'city_King', 'city_Markham', 'city_Mississauga', 'city_Oakville', 'city_Oshawa', 'city_Peel', 'city_Pickering', 'city_Richmond Hill', 'city_Vaughan', 'city_Whitchurch-Stouffville', 'city_York']
Removed: city_Oshawa                              p-value: 0.8263
Removed: city_Pickering                           

In [157]:
#reduced baseline OLS model version 3 lasso + bfs

x_train_version_three = x_train[selected_features].astype(np.float64)
x_test_version_three = x_test[selected_features].astype(np.float64)

ols_reduced = LinearRegression()
ols_reduced.fit(x_train_version_three, y_train)

y_pred_reduced_version_three = ols_reduced.predict(x_test_version_three)

mape = mean_absolute_percentage_error(y_test, y_pred_reduced_version_three) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred_reduced_version_three))
r2 = r2_score(y_test, y_pred_reduced_version_three)

print("reduced OLS model results")
print(f"MAPE {mape:.2f}%")
print(f"RMSE {rmse:,.0f}%")
print(f"r^2: {r2:.4f}")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if mape <= 15 else 'Fail'}")

reduced OLS model results
MAPE 23.38%
RMSE 265,967%
r^2: 0.6500
Target 10% deviation (Zolo benchmark): Fail
Target 15% deviation (sold price): Fail


In [187]:


df_eng['taxes_per_sqft']  = df_eng['taxes'] / df_eng['square_feet_num'].replace(0, np.nan)


df_eng['sqft_per_bedroom']  = df_eng['square_feet_num'] / df_eng['bedrooms'].replace(0, np.nan)

df_eng['is_toronto']  =(df_eng['city_Toronto'] == 1).astype(int)

df_eng['is_peak_season'] = df_eng['listing_month'].isin([4,5,6,7,8]).astype(int)

new_features = ['taxes_per_sqft','sqft_per_bedroom', 'is_toronto', 'is_peak_season']
print(df_eng[new_features].isnull().sum())

taxes_per_sqft      0
sqft_per_bedroom    0
is_toronto          0
is_peak_season      0
dtype: int64


In [188]:


df_eng['taxes_per_sqft']  = df_eng['taxes_per_sqft'].fillna(
    df_eng['taxes_per_sqft'].median()
)
df_eng['sqft_per_bedroom']  = df_eng['sqft_per_bedroom'].fillna(
    df_eng['sqft_per_bedroom'].median()
)

print(df_eng[new_features].isnull().sum())

KEY_COLS_ENG = selected_features + new_features

X_eng = df_eng[KEY_COLS_ENG]
Y_eng = df_eng['sold_price']

x_train_eng, x_test_eng, y_train_eng, y_test_eng = train_test_split(X_eng,Y_eng, test_size = 0.2, random_state=42)

print(f"x_train_eng.shape: {x_train_eng.shape}")
print(f"NaNs in x_train_eng: {x_train_eng.isnull().sum().sum()}")
print(f"NaNs in x_test_eng: {x_test_eng.isnull().sum().sum()}")

nan_check = X_eng.isnull().sum()
print(nan_check[nan_check > 0])

ols_eng = LinearRegression()
ols_eng.fit(x_train_eng, y_train_eng)

y_pred_eng = ols_eng.predict(x_test_eng)

mape = mean_absolute_percentage_error(y_test_eng, y_pred_eng) * 100
rmse = np.sqrt(mean_squared_error(y_test_eng, y_pred_eng))
r2 = r2_score(y_test_eng, y_pred_eng)

print("reduced OLS model results")
print(f"MAPE {mape:.2f}%")
print(f"RMSE {rmse:,.0f}%")
print(f"r^2: {r2:.4f}")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if mape <= 15 else 'Fail'}")

taxes_per_sqft      0
sqft_per_bedroom    0
is_toronto          0
is_peak_season      0
dtype: int64
x_train_eng.shape: (42639, 40)
NaNs in x_train_eng: 0
NaNs in x_test_eng: 0
Series([], dtype: int64)
reduced OLS model results
MAPE 23.37%
RMSE 264,775%
r^2: 0.6531
Target 10% deviation (Zolo benchmark): Fail
Target 15% deviation (sold price): Fail


In [191]:
y_log = np.log1p(df_eng['sold_price'])
x_train_log, x_test_log, y_train_log, y_test_log = train_test_split(df_eng[KEY_COLS_ENG],y_log, test_size = 0.2, random_state=42)

ols_log = LinearRegression()
ols_log.fit(x_train_log, y_train_log)

y_pred_log = np.expm1(ols_log.predict(x_test_log))
y_test_actual = np.expm1(y_test_log)

mape = mean_absolute_percentage_error(y_test_actual, y_pred_log) * 100
r2 = r2_score(y_test_actual, y_pred_log)

print("reduced OLS model results")
print(f"MAPE {mape:.2f}%")
print(f"r^2: {r2:.4f}")


reduced OLS model results
MAPE 18.21%
r^2: 0.6123


In [195]:
#test 3: Ridge with full featture set
scaler_log = StandardScaler()
x_train_log_scaled = scaler_log.fit_transform(x_train_log)
x_test_log_scaled = scaler_log.transform(x_test_log)

alphas_log = [100.0, 500.0, 1000.0, 5000.0]

print(f"{'Alpha':<10} {'MAPE':<12}  {'R^2':<10}")
print("-" * 32)

for alpha in alphas_log:
    ridge_log = Ridge(alpha=alpha)
    ridge_log.fit(x_train_log_scaled, y_train_log)
    y_pred_ridge_log = np.expm1(ridge_log.predict(x_test_log_scaled))

    mape = mean_absolute_percentage_error(y_test_actual, y_pred_ridge_log) * 100
    r2 = r2_score(y_test_actual, y_pred_ridge_log)

    print(f" {alpha:<10} {mape:<12.2f}  {r2: <10.4f}")

Alpha      MAPE          R^2       
--------------------------------
 100.0      18.26         0.6103    
 500.0      18.29         0.6112    
 1000.0     18.31         0.6133    
 5000.0     18.83         0.6193    


With listing price as part of list


In [203]:
# Listing Key columns 

KEY_COLS_V2 = ['listing_price', 'bedrooms', 'bathrooms', 'square_feet_num', 'number_of_storeys', 'latitude', 
            'longitude', 'lot_size_sqft', 'taxes', 'median_income', 'population', 'parking_spaces', 
            'total_rooms', 'bath_bed_ratio', 'listing_year', 'listing_month', 'bedrooms_missing', 'bathrooms_missing',
            'square_feet_num_missing', 'number_of_storeys_missing', 'lot_size_sqft_missing', 'taxes_missing',
            'parking_spaces_missing', 'days_since_sold', 'house_age_encoded', 'property_type_Common Element Condo', 
            'property_type_Condo Apartment', 'property_type_Condo Townhouse', 'property_type_Detached', 'property_type_Link',
            'property_type_Other', 'property_type_Semi-Detached', 'property_type_Vacant Land', 'city_Aurora', 'city_Brampton', 
            'city_Brock', 'city_Burlington', 'city_Caledon', 'city_Clarington', 'city_Durham', 'city_Georgina', 'city_Halton',
            'city_Halton Hills', 'city_King', 'city_Markham', 'city_Milton', 'city_Mississauga', 'city_Oakville', 'city_Oshawa', 
            'city_Peel', 'city_Pickering', 'city_Richmond Hill', 
            'city_Scugog', 'city_Toronto', 'city_Uxbridge', 'city_Vaughan', 'city_Whitby', 'city_Whitchurch-Stouffville', 'city_York']

In [204]:
X_v2 = df_v2[KEY_COLS_V2]
Y_v2 = df_v2['sold_price']

x_train_v2, x_test_v2, y_train_v2, y_test_v2 = train_test_split(X_v2,Y_v2, test_size = 0.2, random_state=42)

print(f"x_train.shape: {x_train_v2.shape}")
print(f"x_test.shape: {x_test_v2.shape}")
print(f"NaNs in x_train: {x_train_v2.isnull().sum().sum()}")
print(f"NaNs in x_test: {x_test_v2.isnull().sum().sum()}")

x_train.shape: (42639, 59)
x_test.shape: (10660, 59)
NaNs in x_train: 0
NaNs in x_test: 0


In [ ]:
#baseline OLS model version 2
ols = LinearRegression()
ols.fit(x_train_v2, y_train_v2)

y_pred_v2 = ols.predict(x_test_v2)

mape = mean_absolute_percentage_error(y_test_v2, y_pred_v2) * 100
rmse = np.sqrt(mean_squared_error(y_test_v2, y_pred_v2))
r2 = r2_score(y_test_v2, y_pred_v2)

print("OlS model results")
print(f"MAPE {mape:.2f}%")
print(f"RMSE {rmse:,.0f}%")
print(f"r^2: {r2:.4f}")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if mape <= 15 else 'Fail'}")

OlS model results
MAPE 3.43%
RMSE 65,599%
r^2: 0.9787
Target 10% deviation (Zolo benchmark): Pass
Target 15% deviation (sold price): Pass


In [212]:
print(f" listing_price in df_eng: {'listing_price' in df_eng.columns}")

 listing_price in df_eng: True


In [213]:
feature_full = [c for c in df_eng.columns if c != 'sold_price']
x_full = df_eng[feature_full]
y_full = df_eng["sold_price"]

x_train_full, x_test_full, y_train_full, y_test_full = train_test_split(x_full,y_full, test_size = 0.2, random_state=42)



ols_full = LinearRegression()
ols_full.fit(x_train_full, y_train_full)

y_pred_train = ols_full.predict(x_train_full)
y_pred_test = ols_full.predict(x_test_full)


train_mape = mean_absolute_percentage_error(y_train_full, y_pred_train) * 100
train_r2 = r2_score(y_train_full, y_pred_train)
test_mape = mean_absolute_percentage_error(y_test_full, y_pred_test) * 100
test_r2 = r2_score(y_test_full, y_pred_test)

print("OlS model results")
print(f"MAPE {train_mape:.2f}%")
print(f"r^2: {train_r2:.4f}")
print(f"MAPE {test_mape:.2f}%")
print(f"r^2: {test_r2:.4f}")
print(f"Gap: {abs(train_mape - test_mape):.2f}%")
print(f"Gap: {abs(train_r2 - test_r2):.2f}%")

#Checking success criteria
print(f"Target 10% deviation (Zolo benchmark): {'Pass' if test_mape <= 10 else 'Fail'}")
print(f"Target 15% deviation (sold price): {'Pass' if test_mape <= 15 else 'Fail'}")


kf = KFold(n_splits =5, shuffle = True, random_state = 42)
cv_mapes = []
cv_r2s = []

for fold, (train_idx, val_idx) in enumerate(kf.split(x_full), 1):
    x_tr, x_val = x_full.iloc[train_idx], x_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

    m = LinearRegression().fit(x_tr, y_tr)
    pred = m.predict(x_val)

    fold_mape = mean_absolute_percentage_error(y_val, pred) * 100
    fold_r2 = r2_score(y_val, pred)

    cv_mapes.append(fold_mape)
    cv_r2s.append(fold_r2)
    print(f"Fold {fold}: MAPE {fold_mape:.2f}% | R^2 = {fold_r2:.4f} ")

print(f"\nMean CV MAPE: {np.mean(cv_mapes):.2f}%  (+/- {np.std(cv_mapes):.2f}%)")
print(f"\nMean CV R^2: {np.mean(cv_r2s):.4f}%  (+/- {np.std(cv_r2s):.4f}%)")

print("Overfitting Diagnosis")
mape_gap = abs(train_mape - test_mape)
if mape_gap < 1.0:
    print(f" Not overfitting - train-test MAPE gap is only {mape_gap:.2f}%")
elif mape_gap < 3.0:
    print(f" Mild  overfitting - train-test MAPE gap is only {mape_gap:.2f}%")
else:
    print(f" Overfitting detected  - train-test MAPE gap is only {mape_gap:.2f}%")
    

cv_std = np.std(cv_mapes)
if cv_std < 0.5:
    print(f" Stable across folds -CV std is only {cv_std:.2f}%")
else:
    print("High variance across folds - CV std is {cv_std:.2f}%")


OlS model results
MAPE 3.41%
r^2: 0.9834
MAPE 3.43%
r^2: 0.9787
Gap: 0.02%
Gap: 0.00%
Target 10% deviation (Zolo benchmark): Pass
Target 15% deviation (sold price): Pass
Fold 1: MAPE 3.43% | R^2 = 0.9787 
Fold 2: MAPE 3.44% | R^2 = 0.9832 
Fold 3: MAPE 3.44% | R^2 = 0.9829 
Fold 4: MAPE 3.45% | R^2 = 0.9838 
Fold 5: MAPE 3.47% | R^2 = 0.9829 

Mean CV MAPE: 3.44%  (+/- 0.01%)

Mean CV R^2: 0.9823%  (+/- 0.0018%)
Overfitting Diagnosis
 Not overfitting - train-test MAPE gap is only 0.02%
 Stable across folds -CV std is only 0.01%


In [214]:
y_log_train = np.log1p(y_train_full)
ols_log_full = LinearRegression()
ols_log_full.fit(x_train_full, y_log_train)
y_pred_log = np.expm1(ols_log_full.predict(x_test_full))

In [215]:
mape_log = mean_absolute_percentage_error(y_test_full, y_pred_log) * 100
print(f"Log-transform MAPE  {mape_log:.2f}%")

Log-transform MAPE  9.70%


In [217]:
scaler = StandardScaler()
x_tr_s = scaler.fit_transform(x_train_full)
x_te_s = scaler.transform(x_test_full)
ridge_full = Ridge(alpha = 10).fit(x_tr_s, y_train_full)
mape_r = mean_absolute_percentage_error(y_test_full, ridge_full.predict(x_te_s)) * 100
print(f"Ridge MAPE: {mape_r:.2f}%")

Ridge MAPE: 3.43%


In [221]:
#The feautrs the model relies on the
importance = pd.DataFrame({
    'feature': x_train_full.columns,
    'coefficient': ols_full.coef_,
    'abs_coef': np.abs(ols_full.coef_)
}).sort_values('abs_coef', ascending = False).head(10)
print(importance)

                        feature   coefficient      abs_coef
38              city_Clarington  73485.392685  73485.392685
22       parking_spaces_missing -70787.107281  70787.107281
21                taxes_missing  63728.049210  63728.049210
34                city_Brampton -63224.164014  63224.164014
47                city_Oakville -62374.654743  62374.654743
44                 city_Markham -40100.549725  40100.549725
40                city_Georgina -38801.718100  38801.718100
57  city_Whitchurch-Stouffville -38563.178921  38563.178921
35                   city_Brock  34232.482424  34232.482424
33                  city_Aurora -31011.735419  31011.735419


In [223]:
importance = pd.DataFrame({
    'feature': x_train_full.columns,
    'coefficient': ols_full.coef_,
    'abs_coef': np.abs(ols_full.coef_)
}).sort_values('abs_coef', ascending = False)
print(importance.to_string(index = False))

                           feature   coefficient     abs_coef
                   city_Clarington  73485.392685 73485.392685
            parking_spaces_missing -70787.107281 70787.107281
                     taxes_missing  63728.049210 63728.049210
                     city_Brampton -63224.164014 63224.164014
                     city_Oakville -62374.654743 62374.654743
                      city_Markham -40100.549725 40100.549725
                     city_Georgina -38801.718100 38801.718100
       city_Whitchurch-Stouffville -38563.178921 38563.178921
                        city_Brock  34232.482424 34232.482424
                       city_Aurora -31011.735419 31011.735419
                 city_Halton Hills -29298.941123 29298.941123
             lot_size_sqft_missing -26091.727901 26091.727901
                       city_Oshawa -25973.470290 25973.470290
                  city_Mississauga -25739.665635 25739.665635
                   city_Burlington -19665.914152 19665.914152
        

In [219]:
print(f"Total features: {len(feature_full)}")

for i, feat in enumerate(feature_full, 1):
    print(f"{i:<4} {feat}")

Total features: 59
1    listing_price
2    bedrooms
3    bathrooms
4    square_feet_num
5    number_of_storeys
6    latitude
7    longitude
8    lot_size_sqft
9    taxes
10   median_income
11   population
12   parking_spaces
13   total_rooms
14   bath_bed_ratio
15   listing_year
16   listing_month
17   bedrooms_missing
18   bathrooms_missing
19   square_feet_num_missing
20   number_of_storeys_missing
21   lot_size_sqft_missing
22   taxes_missing
23   parking_spaces_missing
24   days_since_sold
25   house_age_encoded
26   property_type_Common Element Condo
27   property_type_Condo Apartment
28   property_type_Condo Townhouse
29   property_type_Detached
30   property_type_Link
31   property_type_Other
32   property_type_Semi-Detached
33   property_type_Vacant Land
34   city_Aurora
35   city_Brampton
36   city_Brock
37   city_Burlington
38   city_Caledon
39   city_Clarington
40   city_Durham
41   city_Georgina
42   city_Halton
43   city_Halton Hills
44   city_King
45   city_Markham
46   c